In [1]:
import pandas as pd
import numpy as np

# Read the CSV file
df = pd.read_csv('testdata.csv')

print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]}...")  # Show first 10 columns
df.head()


Original shape: (2237, 62)
Columns: ['participant_id', 'screen', 'x0', 'y0', 'timestamp0', 'x1', 'y1', 'timestamp1', 'x2', 'y2']...


,participant_id,screen,x0,y0,timestamp0,x1,y1,timestamp1,x2,y2,...,timestamp16,x17,y17,timestamp17,x18,y18,timestamp18,x19,y19,timestamp19
0,sample,instruction calibrationText 2,500.20096,499.950160,43:20.2,500.68207,498.951050,43:20.4,506.47742,495.30880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sample,instruction calibrationText 2,849.60095,97.403244,43:21.0,864.59890,112.571430,43:21.1,862.21580,128.66505,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sample,instruction calibrationText 2,715.84990,100.962040,43:22.0,653.53640,87.116516,43:22.1,597.95510,65.78212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sample,instruction calibrationText 2,758.14410,0.000000,43:23.0,801.38100,0.000000,43:23.1,832.04030,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sample,instruction calibrationText 2,850.96783,139.241180,43:24.1,835.99380,143.714840,43:24.2,854.20605,139.40747,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Check unique values in the screen column
print("Unique values in 'screen' column:")
print(f"Total unique values: {df['screen'].nunique()}")
print(f"\nAll unique screen values:")
unique_screens = df['screen'].unique()
for i, screen in enumerate(sorted(unique_screens), 1):
    print(f"{i}. {screen}")
    
print(f"\nValue counts:")
df['screen'].value_counts()


Unique values in 'screen' column:
Total unique values: 423

All unique screen values:
1. accuracy
2. block 1 trial 1 anticipation
3. block 1 trial 1 feedback
4. block 1 trial 1 fixation
5. block 1 trial 10 anticipation
6. block 1 trial 10 feedback
7. block 1 trial 11 anticipation
8. block 1 trial 11 feedback
9. block 1 trial 11 fixation
10. block 1 trial 12 anticipation
11. block 1 trial 12 feedback
12. block 1 trial 13 anticipation
13. block 1 trial 13 feedback
14. block 1 trial 13 fixation
15. block 1 trial 14 anticipation
16. block 1 trial 14 feedback
17. block 1 trial 15 anticipation
18. block 1 trial 15 feedback
19. block 1 trial 2 anticipation
20. block 1 trial 2 feedback
21. block 1 trial 3 anticipation
22. block 1 trial 3 feedback
23. block 1 trial 3 fixation
24. block 1 trial 4 anticipation
25. block 1 trial 4 feedback
26. block 1 trial 4 fixation
27. block 1 trial 5 anticipation
28. block 1 trial 5 feedback
29. block 1 trial 6 anticipation
30. block 1 trial 6 feedback
31. blo

screen
profile uploadPhoto           173
feeling 1                     124
feeling 2                     105
profile bio                    57
accuracy                       49
                             ... 
block 7 trial 6 fixation        1
block 7 trial 10 fixation       1
block 7 trial 15 fixation       1
summary 7 trial 1 fixation      1
block 1 trial 1 fixation        1
Name: count, Length: 423, dtype: int64

In [5]:
# Flatten the data
# Each row will have: participant_id, screen (split into columns), x, y, timestamp

flattened_rows = []

for idx, row in df.iterrows():
    participant_id = row['participant_id']
    screen = row['screen']
    
    # Split screen column: first word is screen_type, rest is screen_id
    if pd.notna(screen):
        screen_parts = str(screen).split()
        screen_type = screen_parts[0] if len(screen_parts) > 0 else ''
        screen_id = ' '.join(screen_parts[1:]) if len(screen_parts) > 1 else ''
    else:
        screen_type = ''
        screen_id = ''
    
    # Find all timestamp columns (timestamp0, timestamp1, etc.)
    timestamp_cols = [col for col in df.columns if col.startswith('timestamp')]
    
    # For each timestamp, extract the corresponding x and y values
    for ts_col in timestamp_cols:
        # Get the index number from the column name (e.g., 'timestamp0' -> 0)
        idx_num = int(ts_col.replace('timestamp', ''))
        
        timestamp = row[ts_col]
        
        # Skip if timestamp is empty/NaN
        if pd.isna(timestamp) or timestamp == '':
            continue
        
        # Get corresponding x and y values
        x_col = f'x{idx_num}'
        y_col = f'y{idx_num}'
        
        x_val = row[x_col]
        y_val = row[y_col]
        
        # Create a new row with screen split into screen_type and screen_id
        new_row = {
            'participant_id': participant_id,
            'screen_type': screen_type,
            'screen_id': screen_id,
            'x': x_val,
            'y': y_val,
            'timestamp': timestamp
        }
        
        flattened_rows.append(new_row)

# Create the flattened dataframe
flattened_df = pd.DataFrame(flattened_rows)

# Reorder columns: participant_id, screen_type, screen_id, x, y, timestamp
column_order = ['participant_id', 'screen_type', 'screen_id', 'x', 'y', 'timestamp']
flattened_df = flattened_df[column_order]

print(f"Flattened shape: {flattened_df.shape}")
print(f"\nColumns: {flattened_df.columns.tolist()}")
flattened_df.head(10)


Flattened shape: (26714, 6)

Columns: ['participant_id', 'screen_type', 'screen_id', 'x', 'y', 'timestamp']


,participant_id,screen_type,screen_id,x,y,timestamp
0,sample,instruction,calibrationText 2,500.20096,499.95016,43:20.2
1,sample,instruction,calibrationText 2,500.68207,498.95105,43:20.4
2,sample,instruction,calibrationText 2,506.47742,495.30880,43:20.5
3,sample,instruction,calibrationText 2,522.05975,484.52933,43:20.5
4,sample,instruction,calibrationText 2,550.51380,466.42136,43:20.6
5,sample,instruction,calibrationText 2,596.45090,422.40952,43:20.6
6,sample,instruction,calibrationText 2,651.83560,355.87634,43:20.7
7,sample,instruction,calibrationText 2,714.62384,274.08673,43:20.8
8,sample,instruction,calibrationText 2,768.75806,199.86055,43:20.8
9,sample,instruction,calibrationText 2,809.58520,128.61832,43:20.9


In [6]:
# Save the flattened data to a new CSV file
flattened_df.to_csv('testdata_flattened.csv', index=False)
print("Flattened data saved to 'testdata_flattened.csv'")

# Display summary
print(f"\nSummary:")
print(f"Original rows: {len(df)}")
print(f"Flattened rows: {len(flattened_df)}")
print(f"\nFirst few rows of flattened data:")
flattened_df.head()


Flattened data saved to 'testdata_flattened.csv'

Summary:
Original rows: 2237
Flattened rows: 26714

First few rows of flattened data:


,participant_id,screen_type,screen_id,x,y,timestamp
0,sample,instruction,calibrationText 2,500.20096,499.95016,43:20.2
1,sample,instruction,calibrationText 2,500.68207,498.95105,43:20.4
2,sample,instruction,calibrationText 2,506.47742,495.30880,43:20.5
3,sample,instruction,calibrationText 2,522.05975,484.52933,43:20.5
4,sample,instruction,calibrationText 2,550.51380,466.42136,43:20.6
